In [92]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

/Users/emmanuel/Documents/belugas/beluga-call-pipeline
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [93]:
import pandas as pd
import glob
import os
pd.set_option('display.max_columns', None)
irene_df = pd.read_csv("../../data/evaluation_snippets/irene/cc_hfpc_20min.Table.1.selections_IR.txt", sep="\t")


In [94]:
# Add 'start_s' indicating the order within each SnippetFilename group (starting at 0)
irene_df['start_s'] = irene_df.groupby('SnippetFilename').cumcount()
irene_df["end_s"] = irene_df["start_s"] + 1

In [95]:
irene_df["BBPC"].value_counts()

BBPC
0    704
1    496
Name: count, dtype: int64

In [96]:
irene_df

,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),ECHO,HFPC,BBPC,Whistle,GROUNDTRUTH,DETAIL,Notes,SnippetFilename,start_s,end_s
0,1,Spectrogram 1,1,0.0,1.0,0.0,96000.0,1,1,0,1,ehw,NaN,NaN,BSM_20170801_18595800.wav,0,1
1,2,Spectrogram 1,1,1.0,2.0,0.0,96000.0,1,1,0,1,ehbw,bm,NaN,BSM_20170801_18595800.wav,1,2
2,3,Spectrogram 1,1,2.0,3.0,0.0,96000.0,1,1,0,1,ehw,hm,NaN,BSM_20170801_18595800.wav,2,3
3,4,Spectrogram 1,1,3.0,4.0,0.0,96000.0,1,0,0,1,ew,NaN,NaN,BSM_20170810_16050029.wav,0,1
4,5,Spectrogram 1,1,4.0,5.0,0.0,96000.0,1,0,0,1,ew,NaN,I this case I would say that the w is not sign...,BSM_20170810_16050029.wav,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,1196,Spectrogram 1,1,1195.0,1196.0,0.0,96000.0,0,1,0,1,hw,hm,NaN,RDL_20200730_08362978.wav,1,2
1196,1197,Spectrogram 1,1,1196.0,1197.0,0.0,96000.0,0,1,0,1,hw,hm,NaN,RDL_20200730_08362978.wav,2,3
1197,1198,Spectrogram 1,1,1197.0,1198.0,0.0,96000.0,1,1,0,1,ehw,hm,NaN,RDL_20200811_08554226.wav,0,1
1198,1199,Spectrogram 1,1,1198.0,1199.0,0.0,96000.0,1,0,0,1,ew,NaN,NaN,RDL_20200811_08554226.wav,1,2


In [97]:
import pandas as pd

# Extract site (first part before the first underscore)
irene_df["Site"] = irene_df["SnippetFilename"].str.split("_").str[0]



In [98]:
old_df = pd.read_csv("../../data/labels/Overlaps_1s.csv")


In [99]:
old_df[["HydrophoneModel", "HydrophoneSensitivity", "SnippetFilename", "Timestamp"]]

,HydrophoneModel,HydrophoneSensitivity,SnippetFilename,Timestamp
0,201359382,-172.7,BSM_20170724_13014300.wav,2017-07-24 13:01:43.00
1,201359382,-172.7,BSM_20170724_13014300.wav,2017-07-24 13:01:43.00
2,201359382,-172.7,BSM_20170724_13014300.wav,2017-07-24 13:01:43.00
3,201359382,-172.7,BSM_20170724_13062370.wav,2017-07-24 13:06:23.70
4,201359382,-172.7,BSM_20170724_13062370.wav,2017-07-24 13:06:23.70
...,...,...,...,...
10928,201359382,-172.7,CAC_20210714_09242300.wav,2021-07-14 09:24:23
10929,201359382,-172.7,CAC_20210714_09242800.wav,2021-07-14 09:24:28
10930,201359382,-172.7,CAC_20210714_09251600.wav,2021-07-14 09:25:16
10931,201359382,-172.7,CAC_20210714_09271800.wav,2021-07-14 09:27:18


In [100]:
# Get unique hydrophone info per snippet (avoid duplicates)
hydrophone_info = old_df[["SnippetFilename", "HydrophoneModel", "HydrophoneSensitivity", "Timestamp"]].drop_duplicates()

# Merge into irene_df
irene_df = irene_df.merge(hydrophone_info, on="SnippetFilename", how="left")

In [101]:
irene_df.rename(columns={"Timestamp": "snippet_start_time"}, inplace=True)
irene_df["snippet_start_time"] = pd.to_datetime(irene_df["snippet_start_time"])


In [102]:
irene_df


,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),ECHO,HFPC,BBPC,Whistle,GROUNDTRUTH,DETAIL,Notes,SnippetFilename,start_s,end_s,Site,HydrophoneModel,HydrophoneSensitivity,snippet_start_time
0,1,Spectrogram 1,1,0.0,1.0,0.0,96000.0,1,1,0,1,ehw,NaN,NaN,BSM_20170801_18595800.wav,0,1,BSM,201359382,-172.7,2017-08-01 18:59:58.000
1,2,Spectrogram 1,1,1.0,2.0,0.0,96000.0,1,1,0,1,ehbw,bm,NaN,BSM_20170801_18595800.wav,1,2,BSM,201359382,-172.7,2017-08-01 18:59:58.000
2,3,Spectrogram 1,1,2.0,3.0,0.0,96000.0,1,1,0,1,ehw,hm,NaN,BSM_20170801_18595800.wav,2,3,BSM,201359382,-172.7,2017-08-01 18:59:58.000
3,4,Spectrogram 1,1,3.0,4.0,0.0,96000.0,1,0,0,1,ew,NaN,NaN,BSM_20170810_16050029.wav,0,1,BSM,201359382,-172.7,2017-08-10 16:05:00.290
4,5,Spectrogram 1,1,4.0,5.0,0.0,96000.0,1,0,0,1,ew,NaN,I this case I would say that the w is not sign...,BSM_20170810_16050029.wav,1,2,BSM,201359382,-172.7,2017-08-10 16:05:00.290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,1196,Spectrogram 1,1,1195.0,1196.0,0.0,96000.0,0,1,0,1,hw,hm,NaN,RDL_20200730_08362978.wav,1,2,RDL,5675,-176.5,2020-07-30 08:36:29.780
1196,1197,Spectrogram 1,1,1196.0,1197.0,0.0,96000.0,0,1,0,1,hw,hm,NaN,RDL_20200730_08362978.wav,2,3,RDL,5675,-176.5,2020-07-30 08:36:29.780
1197,1198,Spectrogram 1,1,1197.0,1198.0,0.0,96000.0,1,1,0,1,ehw,hm,NaN,RDL_20200811_08554226.wav,0,1,RDL,5675,-176.5,2020-08-11 08:55:42.260
1198,1199,Spectrogram 1,1,1198.0,1199.0,0.0,96000.0,1,0,0,1,ew,NaN,NaN,RDL_20200811_08554226.wav,1,2,RDL,5675,-176.5,2020-08-11 08:55:42.260


In [103]:
irene_df["clip_start_time"] = irene_df["snippet_start_time"] + pd.to_timedelta(irene_df["start_s"], unit="s")
irene_df["clip_end_time"] = irene_df["clip_start_time"] + pd.to_timedelta(1, unit="s")

In [79]:
irene_df["clip_filename"] = irene_df["Site"] + "_" + irene_df["clip_start_time"].dt.strftime("%Y%m%d_%H%M%S%f").str[:-4] + ".wav"

# Check for duplicate clip_filenames
# Drop duplicate clip_filenames, keeping the first occurrence
irene_df = irene_df.drop_duplicates(subset="clip_filename", keep="first")

irene_df

,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),ECHO,HFPC,BBPC,Whistle,GROUNDTRUTH,DETAIL,Notes,SnippetFilename,start_s,end_s,Site,HydrophoneModel,HydrophoneSensitivity,snippet_start_time,clip_start_time,clip_end_time,clip_filename
0,1,Spectrogram 1,1,0.0,1.0,0.0,96000.0,1,1,0,1,ehw,NaN,NaN,BSM_20170801_18595800.wav,0,1,BSM,201359382,-172.7,2017-08-01 18:59:58.000,2017-08-01 18:59:58.000,2017-08-01 18:59:59.000,BSM_20170801_18595800.wav
1,2,Spectrogram 1,1,1.0,2.0,0.0,96000.0,1,1,0,1,ehbw,bm,NaN,BSM_20170801_18595800.wav,1,2,BSM,201359382,-172.7,2017-08-01 18:59:58.000,2017-08-01 18:59:59.000,2017-08-01 19:00:00.000,BSM_20170801_18595900.wav
2,3,Spectrogram 1,1,2.0,3.0,0.0,96000.0,1,1,0,1,ehw,hm,NaN,BSM_20170801_18595800.wav,2,3,BSM,201359382,-172.7,2017-08-01 18:59:58.000,2017-08-01 19:00:00.000,2017-08-01 19:00:01.000,BSM_20170801_19000000.wav
3,4,Spectrogram 1,1,3.0,4.0,0.0,96000.0,1,0,0,1,ew,NaN,NaN,BSM_20170810_16050029.wav,0,1,BSM,201359382,-172.7,2017-08-10 16:05:00.290,2017-08-10 16:05:00.290,2017-08-10 16:05:01.290,BSM_20170810_16050029.wav
4,5,Spectrogram 1,1,4.0,5.0,0.0,96000.0,1,0,0,1,ew,NaN,I this case I would say that the w is not sign...,BSM_20170810_16050029.wav,1,2,BSM,201359382,-172.7,2017-08-10 16:05:00.290,2017-08-10 16:05:01.290,2017-08-10 16:05:02.290,BSM_20170810_16050129.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,1196,Spectrogram 1,1,1195.0,1196.0,0.0,96000.0,0,1,0,1,hw,hm,NaN,RDL_20200730_08362978.wav,1,2,RDL,5675,-176.5,2020-07-30 08:36:29.780,2020-07-30 08:36:30.780,2020-07-30 08:36:31.780,RDL_20200730_08363078.wav
1196,1197,Spectrogram 1,1,1196.0,1197.0,0.0,96000.0,0,1,0,1,hw,hm,NaN,RDL_20200730_08362978.wav,2,3,RDL,5675,-176.5,2020-07-30 08:36:29.780,2020-07-30 08:36:31.780,2020-07-30 08:36:32.780,RDL_20200730_08363178.wav
1197,1198,Spectrogram 1,1,1197.0,1198.0,0.0,96000.0,1,1,0,1,ehw,hm,NaN,RDL_20200811_08554226.wav,0,1,RDL,5675,-176.5,2020-08-11 08:55:42.260,2020-08-11 08:55:42.260,2020-08-11 08:55:43.260,RDL_20200811_08554226.wav
1198,1199,Spectrogram 1,1,1198.0,1199.0,0.0,96000.0,1,0,0,1,ew,NaN,NaN,RDL_20200811_08554226.wav,1,2,RDL,5675,-176.5,2020-08-11 08:55:42.260,2020-08-11 08:55:43.260,2020-08-11 08:55:44.260,RDL_20200811_08554326.wav


In [80]:
def set_verif_flags(gt):
    if pd.isna(gt):
        return pd.Series([False, False, False, False])
    gt_str = str(gt)
    if 'a' in gt_str:
        return pd.Series([False, False, False, False])
    return pd.Series([
        'e' in gt_str,  # ECHO_verif
        'b' in gt_str,  # BBPC_verif
        'h' in gt_str,  # HFPC_verif
        'w' in gt_str   # Whislte_verif
    ])

irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = irene_df["GROUNDTRUTH"].apply(set_verif_flags)
# Convert ECHO, BBPC, HFPC, Whistle columns to 0/1 integers
irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]].astype(int)


/var/folders/24/7c98sw1j4_13xl92r115m6640000gn/T/ipykernel_1936/3066683244.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = irene_df["GROUNDTRUTH"].apply(set_verif_flags)
/var/folders/24/7c98sw1j4_13xl92r115m6640000gn/T/ipykernel_1936/3066683244.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]].astype(int)


## Clipping wav files

In [81]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

# Output directory
output_dir = "../../data/Verified_Dataset/clip_wavs/"
os.makedirs(output_dir, exist_ok=True)

snippets_dir = "../../data/Full_Dataset/Snippets_3s_wav/"


grouped = irene_df.groupby(["SnippetFilename"])

for (snippet_filename, ), group in tqdm(grouped, total=len(grouped)):
    # print(snippet_filename)
    source_path = os.path.join(snippets_dir, snippet_filename)
    try:
        # Load the entire snippet once
        y, sr = librosa.load(source_path, sr=None)
        
        # Extract all clips from this snippet
        for idx, row in group.iterrows():
            output_path = os.path.join(output_dir, row["clip_filename"])
            
            # Skip if already exists
            if os.path.exists(output_path):
                continue
            
            # Calculate sample indices
            start_sample = int(row["start_s"] * sr)
            end_sample = int(row["end_s"] * sr)
            
            # Extract and save the clip
            clip = y[start_sample:end_sample]
            sf.write(output_path, clip, sr)
            
    except Exception as e:
        print(f"Error processing {snippet_filename}: {e}")


  0%|          | 0/400 [00:00<?, ?it/s]

100%|██████████| 400/400 [00:05<00:00, 76.91it/s]


In [82]:

irene_df = irene_df.drop(columns=["Selection", "View", "Channel","Begin Time (s)", "End Time (s)", "Low Freq (Hz)", "High Freq (Hz)"])


In [83]:
irene_df["Notes"].value_counts()
irene_df["Boat"] = irene_df["Notes"].str.contains("ship", case=False, na=False).astype(int)


In [84]:
irene_df.to_csv("../../data/Verified_Dataset/labels/labels_irene_20min.csv", index=False)

In [85]:
irene_df[irene_df["clip_filename"] == "BSM_20170801_16441900.wav"]

,ECHO,HFPC,BBPC,Whistle,GROUNDTRUTH,DETAIL,Notes,SnippetFilename,start_s,end_s,Site,HydrophoneModel,HydrophoneSensitivity,snippet_start_time,clip_start_time,clip_end_time,clip_filename,Boat
